# 01 — Mean-CVaR Basics

This notebook walks through the simplest end-to-end Mean-CVaR workflow:

1. Generate a synthetic universe so the playbook runs offline.
2. Build a scenario panel (Gaussian and block bootstrap).
3. Solve Mean-CVaR and compare against Markowitz Max-Sharpe.
4. Visualise weights, expected returns, and tail risk.

**Reference docs:** [docs/MEAN_CVAR.md](../../../docs/MEAN_CVAR.md), [docs/SCENARIO_GENERATION.md](../../../docs/SCENARIO_GENERATION.md).

## 1. Synthetic universe

We use `benchmarks.base.generate_synthetic_dataset` so the notebook is
reproducible without market data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from benchmarks.base import generate_synthetic_dataset

ds = generate_synthetic_dataset(n_assets=10, n_history=504, seed=42)
tickers = [f"A{i:02d}" for i in range(ds.n_assets)]
print(f"Universe: {ds.n_assets} assets, {ds.n_history} days of history")
print(f"Annualised return range: {ds.mu.min():.3f} ... {ds.mu.max():.3f}")
print(f"Annualised vol range:    {np.sqrt(np.diag(ds.Sigma)).min():.3f} ... {np.sqrt(np.diag(ds.Sigma)).max():.3f}")

## 2. Generate scenarios

Mean-CVaR optimisation needs a scenario matrix of shape `(S, n)` —
each row is one realisation of returns across the universe. We try
two methods so you can see how the scenario engine affects the
resulting CVaR.

In [ ]:
from services.scenario_generation import ScenarioConfig, generate_scenarios

gauss = generate_scenarios(
    ds.daily_returns,
    ScenarioConfig(method="gaussian", n_scenarios=5000, seed=42),
)
block = generate_scenarios(
    ds.daily_returns,
    ScenarioConfig(method="block", n_scenarios=5000, block_size=20, seed=42),
)

print(f"Gaussian scenarios: shape={gauss.shape}, worst day={gauss.min():.4f}")
print(f"Block bootstrap:    shape={block.shape}, worst day={block.min():.4f}")

## 3. Solve Mean-CVaR

`run_optimization(..., objective="mean_cvar")` routes through the
solver backend chosen by `services.solver_router`. We pass the
scenario panel and a 30% per-asset cap.

In [ ]:
from core.portfolio_optimizer import run_optimization

result_cvar = run_optimization(
    returns=ds.mu,
    covariance=ds.Sigma,
    objective="mean_cvar",
    scenarios=block,
    asset_names=tickers,
    confidence_level=0.95,
    risk_aversion=1.0,
    weight_min=0.0,
    weight_max=0.30,
)
print(f"Backend:        {result_cvar.backend}")
print(f"Solver:         {result_cvar.solver}")
print(f"Solve time:     {result_cvar.solve_time_ms:.1f} ms")
print(f"Expected return: {result_cvar.expected_return:6.4f}")
print(f"Volatility:     {result_cvar.volatility:6.4f}")
print(f"Sharpe:         {result_cvar.sharpe_ratio:6.4f}")
print(f"VaR 95%:        {result_cvar.var_95:6.4f}")
print(f"CVaR 95%:       {result_cvar.cvar_95:6.4f}")

## 4. Compare against Markowitz

Same universe, different objective. Markowitz maximises Sharpe under
the same weight constraints — but it doesn't see the scenario panel,
only the mean and covariance.

In [ ]:
result_mk = run_optimization(
    returns=ds.mu,
    covariance=ds.Sigma,
    objective="markowitz",
    asset_names=tickers,
    weight_min=0.0,
    weight_max=0.30,
)

# Compute realised tail loss of Markowitz weights on the same scenario panel
mk_losses = -(block @ result_mk.weights)
mk_var = float(np.quantile(mk_losses, 0.95))
mk_cvar = float(mk_losses[mk_losses >= mk_var].mean())

rows = [
    ["Markowitz", result_mk.sharpe_ratio, result_mk.expected_return,
     result_mk.volatility, mk_var, mk_cvar],
    ["Mean-CVaR", result_cvar.sharpe_ratio, result_cvar.expected_return,
     result_cvar.volatility, result_cvar.var_95, result_cvar.cvar_95],
]
header = ["objective", "sharpe", "exp_return", "vol", "var95", "cvar95"]
print("  ".join(f"{h:>10}" for h in header))
for row in rows:
    cells = [row[0]] + [f"{v:.4f}" for v in row[1:]]
    print("  ".join(f"{c:>10}" for c in cells))

## 5. Visualise weights

Mean-CVaR usually keeps the same risk/return targets but tilts away
from assets whose left tails are heavy.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(tickers))
ax.bar(x - 0.20, result_mk.weights, width=0.4, label="Markowitz")
ax.bar(x + 0.20, result_cvar.weights, width=0.4, label="Mean-CVaR")
ax.set_xticks(x)
ax.set_xticklabels(tickers, rotation=0)
ax.set_ylabel("weight")
ax.set_title("Markowitz vs Mean-CVaR weights (synthetic 10-asset universe)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Tail-loss histogram

Plot the realised loss distribution under each portfolio. Mean-CVaR
should give a thinner left tail past the VaR threshold.

In [ ]:
cvar_losses = -(block @ result_cvar.weights)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(mk_losses, bins=60, alpha=0.5, label="Markowitz")
ax.hist(cvar_losses, bins=60, alpha=0.5, label="Mean-CVaR")
ax.axvline(result_cvar.var_95, color="red", linestyle="--", label="CVaR target VaR")
ax.set_xlabel("daily loss (positive = loss)")
ax.set_ylabel("frequency")
ax.set_title("Realised tail-loss distribution on the scenario panel")
ax.legend()
plt.tight_layout()
plt.show()

## Next steps

- **02 — scenario generation:** compare historical / block / Gaussian /
  Student-t methods and see how tail thickness drives CVaR.
- **03 — rebalancing strategies:** turn this static optimisation into
  a backtest with realistic transaction costs.
- **05 — quantum-hybrid comparison:** see where the QUBO-based
  selectors fit alongside Mean-CVaR.